![display relevant image here](path/url/to/image)
- Banner/header image

# Title
- Relevant to Data and Business Context

## Overview
- BLUF (Bottom Line Up Front)
- One paragraph summary of findings and analysis
- Frame your 'story'

## Business Understanding
- Set the stage for analysis
- Why are these findings relevant/important?
- Introduce stakeholders
- Postulate about questions you want to ask/answer

## Data Understanding
- Present the source of data
- Describe the data available
- What is relevant to keep what is not
- Present any data cleaning that needs to happen
- Null values? Type mismatches? Duplicates?

In [2]:
# Imports

import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, isnan

# Create Spark session
spark = SparkSession.builder \
    .appName("Google Play Store Analysis") \
    .getOrCreate()

print("PySpark version:", pyspark.__version__)
print("Spark session created successfully")

PySpark version: 4.2.0
Spark session created successfully


In [3]:
# EDA Code Here - Create New Cells As Needed

df = spark.read.csv(
    "google_play_store_dataset.csv",
    header=True,
    inferSchema=True
)

print("Dataset loaded successfully")

print("Number of rows:", df.count())
print("Number of columns:", len(df.columns))

df.printSchema()

df.show(5, truncate=False)

Dataset loaded successfully
Number of rows: 10841
Number of columns: 13
root
 |-- App: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Rating: string (nullable = true)
 |-- Reviews: string (nullable = true)
 |-- Size: string (nullable = true)
 |-- Installs: string (nullable = true)
 |-- Type: string (nullable = true)
 |-- Price: string (nullable = true)
 |-- Content Rating: string (nullable = true)
 |-- Genres: string (nullable = true)
 |-- Last Updated: string (nullable = true)
 |-- Current Ver: string (nullable = true)
 |-- Android Ver: string (nullable = true)

+--------------------------------------------------+--------------+------+-------+----+-----------+----+-----+--------------+-------------------------+----------------+------------------+------------+
|App                                               |Category      |Rating|Reviews|Size|Installs   |Type|Price|Content Rating|Genres                   |Last Updated    |Current Ver       |Android Ver |
+----

In [4]:
from pyspark.sql.functions import col, sum as spark_sum

null_counts = df.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

null_counts.show(truncate=False)

+---+--------+------+-------+----+--------+----+-----+--------------+------+------------+-----------+-----------+
|App|Category|Rating|Reviews|Size|Installs|Type|Price|Content Rating|Genres|Last Updated|Current Ver|Android Ver|
+---+--------+------+-------+----+--------+----+-----+--------------+------+------------+-----------+-----------+
|0  |0       |0     |0      |0   |0       |0   |0    |1             |0     |0           |1          |1          |
+---+--------+------+-------+----+--------+----+-----+--------------+------+------------+-----------+-----------+



In [5]:
for c in df.columns:
    missing_text = df.filter(
        col(c).isin("NaN", "nan", "N/A", "null", "")
    ).count()
    
    if missing_text > 0:
        print(f"{c}: {missing_text}")

Rating: 1474
Type: 1
Current Ver: 7
Android Ver: 2


In [7]:
# Check duplicate records

total_rows = df.count()
unique_rows = df.dropDuplicates().count()
duplicate_rows = total_rows - unique_rows

print("Total rows:", total_rows)
print("Unique rows:", unique_rows)
print("Duplicate rows:", duplicate_rows)

Total rows: 10841
Unique rows: 10358
Duplicate rows: 483


In [8]:
# Descriptive statistics for the original dataset

df.describe().show(truncate=False)

+-------+--------------------------------------------------+-------------+-----------+-----------------+------------------+------------------+-----+------------------+--------------+------+-----------------+-------------+------------------+
|summary|App                                               |Category     |Rating     |Reviews          |Size              |Installs          |Type |Price             |Content Rating|Genres|Last Updated     |Current Ver  |Android Ver       |
+-------+--------------------------------------------------+-------------+-----------+-----------------+------------------+------------------+-----+------------------+--------------+------+-----------------+-------------+------------------+
|count  |10841                                             |10841        |10841      |10841            |10841             |10841             |10841|10841             |10840         |10841 |10841            |10840        |10840             |
|mean   |NULL                       

In [9]:
# Check important categorical value distributions

df.groupBy("Type").count().orderBy("count", ascending=False).show()

df.groupBy("Content Rating").count().orderBy(
    "count", ascending=False
).show()

df.groupBy("Category").count().orderBy(
    "count", ascending=False
).show(40, truncate=False)

+------+-----+
|  Type|count|
+------+-----+
|  Free|10037|
|  Paid|  800|
|     0|    1|
|102248|    1|
|   NaN|    1|
|  2509|    1|
+------+-----+

+---------------+-----+
| Content Rating|count|
+---------------+-----+
|       Everyone| 8713|
|           Teen| 1208|
|     Mature 17+|  498|
|   Everyone 10+|  414|
|Adults only 18+|    3|
|        Unrated|    2|
|     5,000,000+|    1|
|           NULL|    1|
|     1,000,000+|    1|
+---------------+-----+

+-------------------+-----+
|Category           |count|
+-------------------+-----+
|FAMILY             |1972 |
|GAME               |1144 |
|TOOLS              |843  |
|MEDICAL            |463  |
|BUSINESS           |460  |
|PRODUCTIVITY       |424  |
|PERSONALIZATION    |392  |
|COMMUNICATION      |387  |
|SPORTS             |384  |
|LIFESTYLE          |382  |
|FINANCE            |366  |
|HEALTH_AND_FITNESS |340  |
|PHOTOGRAPHY        |335  |
|SOCIAL             |295  |
|NEWS_AND_MAGAZINES |283  |
|SHOPPING           |260  |
|TRA

In [10]:
# Identify suspicious records where Type is not Free or Paid

df.filter(
    ~col("Type").isin("Free", "Paid")
).show(truncate=False)

+---------------------------------------+-------------+-----------+-------------+-------------------+--------+------+------------------+--------------+-----------------+-------------+------------------+------------------+
|App                                    |Category     |Rating     |Reviews      |Size               |Installs|Type  |Price             |Content Rating|Genres           |Last Updated |Current Ver       |Android Ver       |
+---------------------------------------+-------------+-----------+-------------+-------------------+--------+------+------------------+--------------+-----------------+-------------+------------------+------------------+
|"Yanosik: ""antyradar""                | traffic jams| navigation| camera"     |MAPS_AND_NAVIGATION|4.4     |102248|Varies with device|5,000,000+    |Free             |0            |Everyone          |Maps & Navigation |
|"Women""s Health Tips(Breast           |Face         |Body       |weight lose)"|HEALTH_AND_FITNESS |4.2     |25

In [12]:
# Check suspicious ratings

from pyspark.sql.functions import expr

df.select(
    "App",
    "Category",
    "Rating",
    "Reviews",
    "Type",
    "Price"
).withColumn(
    "Rating_num",
    expr("try_cast(Rating as double)")
).filter(
    col("Rating").isNotNull() &
    (
        col("Rating_num").isNull() |
        (col("Rating_num") < 0) |
        (col("Rating_num") > 5)
    )
).show(truncate=False)

+-------------------------------------------------------------------------------+-------------------+------+-------+----+-----+----------+
|App                                                                            |Category           |Rating|Reviews|Type|Price|Rating_num|
+-------------------------------------------------------------------------------+-------------------+------+-------+----+-----+----------+
|Mcqueen Coloring pages                                                         |ART_AND_DESIGN     |NaN   |61     |Free|0    |NaN       |
|Wrinkles and rejuvenation                                                      |BEAUTY             |NaN   |182    |Free|0    |NaN       |
|Manicure - nail design                                                         |BEAUTY             |NaN   |119    |Free|0    |NaN       |
|Skin Care and Natural Beauty                                                   |BEAUTY             |NaN   |654    |Free|0    |NaN       |
|Secrets of beauty, youth a

## Data Preparation



In [13]:
# Data Preparation - Initial Cleaning

from pyspark.sql.functions import col, regexp_replace, to_date, expr

# 1. Remove exact duplicate rows
df_clean = df.dropDuplicates()

# 2. Keep only valid app types
df_clean = df_clean.filter(
    col("Type").isin("Free", "Paid")
)

# 3. Convert text "NaN" to actual NULL for Rating
df_clean = df_clean.withColumn(
    "Rating",
    expr("try_cast(Rating as double)")
)

# 4. Convert Reviews to numeric
df_clean = df_clean.withColumn(
    "Reviews",
    expr("try_cast(Reviews as long)")
)

# 5. Remove + and commas from Installs, then convert to numeric
df_clean = df_clean.withColumn(
    "Installs",
    regexp_replace(col("Installs"), "[+,]", "").cast("long")
)

# 6. Remove $ from Price, then convert to numeric
df_clean = df_clean.withColumn(
    "Price",
    regexp_replace(col("Price"), r"\$", "").cast("double")
)

# 7. Convert Last Updated to date
df_clean = df_clean.withColumn(
    "Last Updated",
    to_date(col("Last Updated"), "MMMM d, yyyy")
)

print("Original rows:", df.count())
print("Rows after initial cleaning:", df_clean.count())

df_clean.printSchema()

Original rows: 10841
Rows after initial cleaning: 10354
root
 |-- App: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Rating: double (nullable = true)
 |-- Reviews: long (nullable = true)
 |-- Size: string (nullable = true)
 |-- Installs: long (nullable = true)
 |-- Type: string (nullable = true)
 |-- Price: double (nullable = true)
 |-- Content Rating: string (nullable = true)
 |-- Genres: string (nullable = true)
 |-- Last Updated: date (nullable = true)
 |-- Current Ver: string (nullable = true)
 |-- Android Ver: string (nullable = true)



In [14]:
# Validate cleaned dataset

print("Original rows:", df.count())
print("Cleaned rows:", df_clean.count())
print("Cleaned columns:", len(df_clean.columns))

# Check duplicates remaining
duplicates_remaining = df_clean.count() - df_clean.dropDuplicates().count()
print("Duplicates remaining:", duplicates_remaining)

# Check important numeric columns
df_clean.select(
    "Rating",
    "Reviews",
    "Installs",
    "Price",
    "Last Updated"
).printSchema()

# Preview cleaned data
df_clean.select(
    "App",
    "Category",
    "Rating",
    "Reviews",
    "Installs",
    "Type",
    "Price",
    "Last Updated"
).show(10, truncate=False)

Original rows: 10841
Cleaned rows: 10354
Cleaned columns: 13
Duplicates remaining: 0
root
 |-- Rating: double (nullable = true)
 |-- Reviews: long (nullable = true)
 |-- Installs: long (nullable = true)
 |-- Price: double (nullable = true)
 |-- Last Updated: date (nullable = true)

+-------------------------------------------------+-----------------+------+-------+---------+----+-----+------------+
|App                                              |Category         |Rating|Reviews|Installs |Type|Price|Last Updated|
+-------------------------------------------------+-----------------+------+-------+---------+----+-----+------------+
|Free VIN Report for Used Cars                    |AUTO_AND_VEHICLES|4.6   |2431   |100000   |Free|0.0  |2018-06-12  |
|Opera Browser: Fast and Secure                   |COMMUNICATION    |4.4   |2473509|100000000|Free|0.0  |2018-07-31  |
|Calls & Text by Mo+                              |COMMUNICATION    |4.2   |83239  |5000000  |Free|0.0  |2018-04-19  |
|Th

In [15]:
# Data Prep Code Here
from pyspark.sql.functions import col, sum as spark_sum

missing_clean = df_clean.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in df_clean.columns
])

missing_clean.show(truncate=False)

+---+--------+------+-------+----+--------+----+-----+--------------+------+------------+-----------+-----------+
|App|Category|Rating|Reviews|Size|Installs|Type|Price|Content Rating|Genres|Last Updated|Current Ver|Android Ver|
+---+--------+------+-------+----+--------+----+-----+--------------+------+------------+-----------+-----------+
|0  |0       |0     |0      |0   |0       |0   |0    |0             |0     |0           |1          |0          |
+---+--------+------+-------+----+--------+----+-----+--------------+------+------------+-----------+-----------+



In [24]:
# Save cleaned data file for Tableau usage
df_clean.toPandas().to_csv(
    "google_play_store_cleaned.csv",
    index=False
)

print("Final cleaned dataset exported successfully.")
print("Rows:", df_clean.count())
print("File: google_play_store_cleaned.csv")

Final cleaned dataset exported successfully.
Rows: 10354
File: google_play_store_cleaned.csv


## Data Analysis

In [17]:
# Analysis Code Here - if needed
# Analysis 1: App performance by category

from pyspark.sql.functions import avg, sum as spark_sum, count

category_analysis = df_clean.groupBy("Category").agg(
    count("App").alias("Number_of_Apps"),
    avg("Rating").alias("Average_Rating"),
    spark_sum("Installs").alias("Total_Installs")
).orderBy("Total_Installs", ascending=False)

category_analysis.show(20, truncate=False)

+-------------------+--------------+-----------------+--------------+
|Category           |Number_of_Apps|Average_Rating   |Total_Installs|
+-------------------+--------------+-----------------+--------------+
|GAME               |1121          |NaN              |31544024415   |
|COMMUNICATION      |366           |NaN              |24152276251   |
|SOCIAL             |280           |NaN              |12513867902   |
|PRODUCTIVITY       |407           |NaN              |12463091369   |
|TOOLS              |843           |NaN              |11452771915   |
|FAMILY             |1942          |NaN              |10041692505   |
|PHOTOGRAPHY        |322           |NaN              |9721247655    |
|TRAVEL_AND_LOCAL   |237           |NaN              |6361887146    |
|VIDEO_PLAYERS      |175           |NaN              |6222002720    |
|NEWS_AND_MAGAZINES |264           |NaN              |5393217760    |
|SHOPPING           |224           |NaN              |2573348785    |
|ENTERTAINMENT      

In [18]:
from pyspark.sql.functions import col, when, isnan

# Replace NaN ratings with null
df_clean = df_clean.withColumn(
    "Rating",
    when(isnan(col("Rating")), None)
    .otherwise(col("Rating"))
)

# Re-run category analysis
category_analysis = df_clean.groupBy("Category").agg(
    count("App").alias("Number_of_Apps"),
    avg("Rating").alias("Average_Rating"),
    spark_sum("Installs").alias("Total_Installs")
).orderBy("Total_Installs", ascending=False)

category_analysis.show(20, truncate=False)

+-------------------+--------------+------------------+--------------+
|Category           |Number_of_Apps|Average_Rating    |Total_Installs|
+-------------------+--------------+------------------+--------------+
|GAME               |1121          |4.281284916201114 |31544024415   |
|COMMUNICATION      |366           |4.1514657980456   |24152276251   |
|SOCIAL             |280           |4.254918032786883 |12513867902   |
|PRODUCTIVITY       |407           |4.201796407185629 |12463091369   |
|TOOLS              |843           |4.047411444141692 |11452771915   |
|FAMILY             |1942          |4.191152502910362 |10041692505   |
|PHOTOGRAPHY        |322           |4.182894736842105 |9721247655    |
|TRAVEL_AND_LOCAL   |237           |4.094146341463415 |6361887146    |
|VIDEO_PLAYERS      |175           |4.063749999999997 |6222002720    |
|NEWS_AND_MAGAZINES |264           |4.128504672897197 |5393217760    |
|SHOPPING           |224           |4.2514851485148535|2573348785    |
|ENTER

In [19]:
# Analysis 2: Compare Free vs Paid apps

type_analysis = df_clean.groupBy("Type").agg(
    count("App").alias("Number_of_Apps"),
    avg("Rating").alias("Average_Rating"),
    spark_sum("Installs").alias("Total_Installs"),
    avg("Price").alias("Average_Price")
).orderBy("Total_Installs", ascending=False)

type_analysis.show(truncate=False)

+----+--------------+-----------------+--------------+------------------+
|Type|Number_of_Apps|Average_Rating   |Total_Installs|Average_Price     |
+----+--------------+-----------------+--------------+------------------+
|Free|9589          |4.182397003745319|146556688646  |0.0               |
|Paid|765           |4.261500815660685|69225881      |13.955555555555394|
+----+--------------+-----------------+--------------+------------------+



In [20]:
# Analysis 3: Top 10 most installed apps

top_apps = df_clean.select(
    "App",
    "Category",
    "Installs",
    "Rating",
    "Reviews",
    "Type"
).orderBy(
    col("Installs").desc()
)

top_apps.show(10, truncate=False)

+-----------------+------------------+----------+------+--------+----+
|App              |Category          |Installs  |Rating|Reviews |Type|
+-----------------+------------------+----------+------+--------+----+
|Google News      |NEWS_AND_MAGAZINES|1000000000|3.9   |878065  |Free|
|Instagram        |SOCIAL            |1000000000|4.5   |66509917|Free|
|Subway Surfers   |GAME              |1000000000|4.5   |27723193|Free|
|Google News      |NEWS_AND_MAGAZINES|1000000000|3.9   |877643  |Free|
|Facebook         |SOCIAL            |1000000000|4.1   |78158306|Free|
|Facebook         |SOCIAL            |1000000000|4.1   |78128208|Free|
|Google Play Games|FAMILY            |1000000000|4.3   |7168735 |Free|
|Google Photos    |PHOTOGRAPHY       |1000000000|4.5   |10858556|Free|
|Google Play Games|ENTERTAINMENT     |1000000000|4.3   |7165362 |Free|
|Hangouts         |COMMUNICATION     |1000000000|4.0   |3419513 |Free|
+-----------------+------------------+----------+------+--------+----+
only s

In [21]:
# Analysis 4: Top 10 most reviewed apps

top_reviewed_apps = df_clean.select(
    "App",
    "Category",
    "Reviews",
    "Rating",
    "Installs",
    "Type"
).orderBy(
    col("Reviews").desc()
)

top_reviewed_apps.show(10, truncate=False)

+----------------------------------------+-------------+--------+------+----------+----+
|App                                     |Category     |Reviews |Rating|Installs  |Type|
+----------------------------------------+-------------+--------+------+----------+----+
|Facebook                                |SOCIAL       |78158306|4.1   |1000000000|Free|
|Facebook                                |SOCIAL       |78128208|4.1   |1000000000|Free|
|WhatsApp Messenger                      |COMMUNICATION|69119316|4.4   |1000000000|Free|
|WhatsApp Messenger                      |COMMUNICATION|69109672|4.4   |1000000000|Free|
|Instagram                               |SOCIAL       |66577446|4.5   |1000000000|Free|
|Instagram                               |SOCIAL       |66577313|4.5   |1000000000|Free|
|Instagram                               |SOCIAL       |66509917|4.5   |1000000000|Free|
|Messenger – Text and Video Chat for Free|COMMUNICATION|56646578|4.0   |1000000000|Free|
|Messenger – Text and

In [22]:
# Analysis 5: Average rating by category

rating_by_category = df_clean.groupBy("Category").agg(
    count("App").alias("Number_of_Apps"),
    avg("Rating").alias("Average_Rating")
).filter(
    col("Average_Rating").isNotNull()
).orderBy(
    col("Average_Rating").desc()
)

rating_by_category.show(20, truncate=False)

+-------------------+--------------+------------------+
|Category           |Number_of_Apps|Average_Rating    |
+-------------------+--------------+------------------+
|EVENTS             |64            |4.435555555555556 |
|EDUCATION          |130           |4.375968992248063 |
|ART_AND_DESIGN     |65            |4.358064516129031 |
|BOOKS_AND_REFERENCE|230           |4.347457627118645 |
|PERSONALIZATION    |388           |4.333870967741935 |
|PARENTING          |60            |4.300000000000001 |
|GAME               |1121          |4.281284916201114 |
|BEAUTY             |53            |4.2785714285714285|
|HEALTH_AND_FITNESS |305           |4.26168582375479  |
|SOCIAL             |280           |4.254918032786883 |
|SHOPPING           |224           |4.2514851485148535|
|WEATHER            |82            |4.243999999999999 |
|SPORTS             |351           |4.225174825174825 |
|PRODUCTIVITY       |407           |4.201796407185629 |
|FAMILY             |1942          |4.1911525029

In [23]:
# Analysis 6: Relationship between Reviews and Ratings

reviews_ratings = df_clean.select(
    "App",
    "Category",
    "Reviews",
    "Rating",
    "Installs"
).filter(
    col("Rating").isNotNull()
).orderBy(
    col("Reviews").desc()
)

reviews_ratings.show(20, truncate=False)

+----------------------------------------+-------------+--------+------+----------+
|App                                     |Category     |Reviews |Rating|Installs  |
+----------------------------------------+-------------+--------+------+----------+
|Facebook                                |SOCIAL       |78158306|4.1   |1000000000|
|Facebook                                |SOCIAL       |78128208|4.1   |1000000000|
|WhatsApp Messenger                      |COMMUNICATION|69119316|4.4   |1000000000|
|WhatsApp Messenger                      |COMMUNICATION|69109672|4.4   |1000000000|
|Instagram                               |SOCIAL       |66577446|4.5   |1000000000|
|Instagram                               |SOCIAL       |66577313|4.5   |1000000000|
|Instagram                               |SOCIAL       |66509917|4.5   |1000000000|
|Messenger – Text and Video Chat for Free|COMMUNICATION|56646578|4.0   |1000000000|
|Messenger – Text and Video Chat for Free|COMMUNICATION|56642847|4.0   |1000

### Link to Published Dashboard

In [ ]:
Tableau Public Dashboard:
https://public.tableau.com/app/profile/reuben.mutisya/viz/GoogleplayDashboard_17867222217600/GooglePlayStoreAnalysisDashboard?publish=yes

## Conclusion

Markdown here

The analysis of the Google Play Store dataset provided useful insights into app categories, installations, ratings, reviews, and pricing.

The main findings from the analysis are:

- **Games and Communication apps** are among the categories with the highest total number of installations, showing strong user demand in these areas.
- Most applications in the dataset are **free apps**, with 9,589 free apps compared with 765 paid apps.
- The average ratings across many app categories are generally above **4.0**, indicating relatively high levels of user satisfaction.
- Popular applications such as **Subway Surfers, Google Photos, and Hangouts** appear among the most installed applications.
- Applications such as **Instagram and Facebook** are among the most reviewed apps, demonstrating high levels of user engagement.
- The interactive Tableau dashboard allows users to filter the results by category and explore differences in installs, ratings, and reviews.

Overall, the analysis shows that app popularity is influenced by both the number of installations and the level of user engagement. Free apps dominate the Google Play Store dataset, while categories such as Games and Communication attract particularly large numbers of users.

The combination of Python, Pandas, PySpark, and Tableau provided an effective workflow for cleaning, analyzing, and visualizing the dataset.